<a href="https://colab.research.google.com/github/Richaweb/DS/blob/main/Recommendation_system_osean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Collaborative Analysis Using KNN

In [ ]:
#pip install scikit -learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
anime = pd.read_csv('/content/anime.csv')
rating = pd.read_csv('/content/rating.csv')

In [ ]:
anime.shape

(12294, 7)

In [ ]:
anime.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [ ]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [ ]:
anime.describe(include='all')

,anime_id,name,genre,type,episodes,rating,members
count,12294.000000,12294,12232,12269,12294,12064.000000,1.229400e+04
unique,NaN,12292,3264,6,187,NaN,NaN
top,NaN,Saru Kani Gassen,Hentai,TV,1,NaN,NaN
freq,NaN,2,823,3787,5677,NaN,NaN
mean,14058.221653,NaN,NaN,NaN,NaN,6.473902,1.807134e+04
std,11455.294701,NaN,NaN,NaN,NaN,1.026746,5.482068e+04
min,1.000000,NaN,NaN,NaN,NaN,1.670000,5.000000e+00
25%,3484.250000,NaN,NaN,NaN,NaN,5.880000,2.250000e+02
50%,10260.500000,NaN,NaN,NaN,NaN,6.570000,1.550000e+03
75%,24794.500000,NaN,NaN,NaN,NaN,7.180000,9.437000e+03


In [ ]:
rating.shape

(7813737, 3)

In [ ]:
rating.describe()

,user_id,anime_id,rating
count,7.813737e+06,7.813737e+06,7.813737e+06
mean,3.672796e+04,8.909072e+03,6.144030e+00
std,2.099795e+04,8.883950e+03,3.727800e+00
min,1.000000e+00,1.000000e+00,-1.000000e+00
25%,1.897400e+04,1.240000e+03,6.000000e+00
50%,3.679100e+04,6.213000e+03,7.000000e+00
75%,5.475700e+04,1.409300e+04,9.000000e+00
max,7.351600e+04,3.451900e+04,1.000000e+01


In [ ]:
rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7813737 entries, 0 to 7813736
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 178.8 MB


In [ ]:
rating = rating[rating['rating'] != -1]
rating.shape

(6337241, 3)

In [ ]:
#check for null value in anime dataset.

In [ ]:
anime.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [ ]:
#handle null values
anime = anime.drop(list(anime.loc[anime['rating'].isnull()==True].index),axis=0)
anime = anime.fillna('Unknown')
anime = anime.reset_index(drop=True)

In [ ]:
anime.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


In [ ]:
anime['genre'].unique()

array(['Drama, Romance, School, Supernatural',
       'Action, Adventure, Drama, Fantasy, Magic, Military, Shounen',
       'Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen',
       ..., 'Action, Comedy, Hentai, Romance, Supernatural',
       'Hentai, Sports', 'Hentai, Slice of Life'], dtype=object)

In [ ]:
anime['type'].unique()

array(['Movie', 'TV', 'OVA', 'Special', 'Music', 'ONA'], dtype=object)

#check for duplicate users

In [ ]:
sampled_users=rating['user_id'].drop_duplicates().sample(n=10000,random_state=40).values
filter_rating=rating[rating['user_id'].isin(sampled_users)]

In [ ]:
user_index ={user: index for index, user in enumerate(filter_rating['user_id'].unique())}
anime_index ={anime: index for index, anime in enumerate(filter_rating['anime_id'].unique())}


rows = filter_rating['user_id'].map(user_index).astype(np.int64).values
col = filter_rating['user_id'].map(user_index).astype(np.int64).values
values = filter_rating['rating'].astype(np.float64).values

In [ ]:
sparse_matrix = csr_matrix((values, (rows, col)))

#Model Training

In [ ]:
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=15)
knn.fit(sparse_matrix.tocsr())

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=15)

In [ ]:
#random users lists
random_user = np.random.randint(0, sparse_matrix.shape[0])
random_user_vector = sparse_matrix.getrow(random_user).toarray().reshape(1,-1)

In [ ]:
#similar users
distances, indices = knn.kneighbors(random_user_vector, n_neighbors=15)
similar_users = indices[0]

#Get Random Users

In [ ]:
random_user_id = list(user_index.keys())[random_user]
user_anime = set(filter_rating[filter_rating['user_id'] == random_user_id]['anime_id'])

In [ ]:
recommendations_anime = set()

for user in similar_users:
    user_id = list(user_index.keys())[user]
    recommendations_anime.update(filter_rating[filter_rating['user_id'] == user_id]['anime_id'].values)


recommendations_anime.difference_update(user_anime)


In [ ]:
recommendations_anime_names = anime[anime['anime_id'].isin(recommendations_anime)]['name']

In [ ]:
print('Anime Recommendation:')
for i, anime_name in enumerate(recommendations_anime_names,1):
  print(f'{i}. {anime_name}')

Anime Recommendation:
1. Fullmetal Alchemist: Brotherhood
2. Steins;Gate
3. Gintama&#039;
4. Gintama&#039;: Enchousen
5. Clannad: After Story
6. Gintama
7. Code Geass: Hangyaku no Lelouch R2
8. Sen to Chihiro no Kamikakushi
9. Ookami Kodomo no Ame to Yuki
10. Code Geass: Hangyaku no Lelouch
11. Hajime no Ippo
12. Rurouni Kenshin: Meiji Kenkaku Romantan - Tsuioku-hen
13. Cowboy Bebop
14. One Punch Man
15. Mononoke Hime
16. Suzumiya Haruhi no Shoushitsu
17. Monogatari Series: Second Season
18. Mushishi
19. Great Teacher Onizuka
20. Hajime no Ippo: New Challenger
21. Howl no Ugoku Shiro
22. Fate/Zero 2nd Season
23. Monster
24. Bakuman. 3rd Season
25. Death Note
26. Hajime no Ippo: Rising
27. Kara no Kyoukai 5: Mujun Rasen
28. Natsume Yuujinchou San
29. Boku dake ga Inai Machi
30. Aria The Origination
31. Tengen Toppa Gurren Lagann Movie: Lagann-hen
32. Zoku Natsume Yuujinchou
33. Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai.
34. Steins;Gate Movie: Fuka Ryouiki no Déjà vu
35. Ho